# 5.1 k-Nearest Neighbors Imputation (VAE-Aligned & Support für unvollständige Daten)

Dieses Notebook führt eine Imputation mittels kNN (k-Nearest Neighbors) durch. Es ist auf die VAE-Pipeline abgestimmt und unterstützt die Auswertung unvollständiger Proben.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from pathlib import Path
import os
import json
import itertools
import random
import joblib

In [ ]:
# ------------------------- Parameter -------------------------
if 'TARGET_RUN_INDEX' not in globals(): TARGET_RUN_INDEX = 0
if 'K_NEIGHBORS' not in globals(): K_NEIGHBORS = 5
if 'INCLUDE_INCOMPLETE' not in globals(): INCLUDE_INCOMPLETE = False

In [ ]:
# ------------------------- Daten-Laden -------------------------
def get_latest_preprocessing_file():
    potential_roots = [
        Path.cwd().parent,
        Path('f:/Abschlussarbeit/Abschlussarbeit Bearbeitung/Jupyter Notebooks')
    ]
    
    for root in potential_roots:
        prep_path = root / '3_Machine-Learning' / '3.1_Preprocessing' / 'Preprocessing'
        if prep_path.exists():
            timestamp_folders = [f for f in prep_path.iterdir() if f.is_dir()]
            if timestamp_folders:
                latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
                target_file = latest_folder / 'Preprocessed_SOM_Ready.csv'
                if target_file.exists():
                    return target_file
    return None

csv_path = get_latest_preprocessing_file()
if csv_path is None:
    raise FileNotFoundError("Konnte 'Preprocessed_SOM_Ready.csv' in den erwarteten Verzeichnissen nicht finden.")

df_full = pd.read_csv(csv_path)
print(f"Geladen: {len(df_full)} Zeilen aus {csv_path.parent.name}")

In [ ]:
# ------------------------- Feature-Selektion -------------------------
FIXED_BASE_FEATURES = [
    "Na_in_mmol/L", "Mg_in_mmol/L", "Ca_in_mmol/L", 
    "Cl_in_mmol/L", "SO4_in_mmol/L", "HCO3_in_mmol/L"
]

RUNS = [
    {"name": "Base_without_pH", "add": []},
    {"name": "Base_with_pH",    "add": ["pH"]},
    {"name": "Plus_pH-Fe",      "add": ["pH", "Fe_in_mmol/L"]},
    {"name": "Plus_pH-K-Fe",    "add": ["pH", "K_in_mmol/L", "Fe_in_mmol/L"]},
    {"name": "Plus_pH-K-Fe-Mn", "add": ["pH", "K_in_mmol/L", "Fe_in_mmol/L", "Mn_in_mmol/L"]},
    {"name": "Plus_temperature", "add": ["temperature_in_c"]}
]

def get_training_features(user_selection, df_columns):
    training_features = []
    for user_col in user_selection: 
        found = False
        candidates = [c for c in df_columns if c.startswith(user_col) and "_gauss" in c]
        if candidates:
            best_match = next((c for c in candidates if "_log_gauss" in c), candidates[0])
            training_features.append(best_match)
            found = True
        else:
            if user_col in df_columns:
                training_features.append(user_col)
                found = True
        
        if not found:
             fuzzy = [c for c in df_columns if c.startswith(user_col) and "_gauss" in c]
             if fuzzy:
                 best_match = fuzzy[0]
                 training_features.append(best_match)
                 found = True
    return training_features

selected_run = RUNS[TARGET_RUN_INDEX]
features = get_training_features(FIXED_BASE_FEATURES + selected_run['add'], df_full.columns)
print(f"Ausgewählte Features für {selected_run['name']}: {features}")

In [ ]:
# ------------------------- Imputation & Evaluierung -------------------------
if INCLUDE_INCOMPLETE:
    # ------------------------- Mindestens ein Wert muss vorhanden sein -------------------------
    data_subset = df_full[features].dropna(how='all').copy()
    mode_str = "Unvollständige Proben enthalten"
else:
    data_subset = df_full[features].dropna().copy()
    mode_str = "Nur vollständige Proben"

X_original_full = data_subset.values
print(f"Modus: {mode_str} | Daten-Größe: {len(X_original_full)}")

# ------------------------- Aufteilung angleichen -------------------------
X_train_orig, X_test_orig = train_test_split(
    X_original_full, test_size=0.1, random_state=42
)

# ------------------------- Skalierung -------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_orig)
X_test_scaled = scaler.transform(X_test_orig)

# ------------------------- kNN Imputer - Fit auf dem Train-Set -------------------------
imputer = KNNImputer(n_neighbors=K_NEIGHBORS)
imputer.fit(X_train_scaled)

output_dir = Path("Inference_Results") / f"kNN_Run_{TARGET_RUN_INDEX:03d}"
output_dir.mkdir(parents=True, exist_ok=True)

num_features = len(features)
for k in range(1, num_features):
    print(f"Verarbeite Level {k}... ")
    combos = list(itertools.combinations(range(num_features), k))
    if len(combos) > 50: combos_to_run = random.sample(combos, 50)
    else: combos_to_run = combos

    all_results = []
    for combo_indices in combos_to_run:
        X_target_scaled = X_test_scaled.copy()
        X_target_orig = X_test_orig.copy()
        
        # ------------------------- Maskierung -------------------------
        is_masked = np.zeros(X_target_scaled.shape, dtype=bool)
        for f_idx in combo_indices:
            # ------------------------- Nur maskieren, wenn ein Wert vorhanden war-------------------------
            mask_condition = ~np.isnan(X_target_scaled[:, f_idx])
            X_target_scaled[mask_condition, f_idx] = np.nan
            is_masked[mask_condition, f_idx] = True
        
        # ------------------------- Imputieren -------------------------
        X_imputed_scaled = imputer.transform(X_target_scaled)
        X_imputed = scaler.inverse_transform(X_imputed_scaled)
        
        # ------------------------- Nur maskierte Werte für die Evaluierung sammeln -------------------------
        for f_idx in combo_indices:
             mask = is_masked[:, f_idx]
             if not mask.any(): continue
             
             feat_name = features[f_idx]
             orig_masked = X_target_orig[mask, f_idx]
             imp_masked = X_imputed[mask, f_idx]
             
             for o, im in zip(orig_masked, imp_masked):
                 all_results.append({
                     "Feature": feat_name,
                     "Original": o,
                     "Imputed": im,
                     "Split": "Test",
                     "Masking_Level": k,
                     "Masked_Combination": str([features[i] for i in combo_indices])
                 })
    
    if all_results:
        df_res = pd.DataFrame(all_results)
        df_res.to_csv(output_dir / f"Imputation_Results_kNN_Level_{k}.csv", index=False, float_format='%.4f')
print("Fertig.")